In [ ]:
import os
import shutil
import cv2
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from concurrent.futures import ThreadPoolExecutor, as_completed

# Prevenim blocarea thread-urilor OpenCV
cv2.setNumThreads(0)

# ==========================================
# 1. CONFIGURARE CAI SI PARAMETRI
# ==========================================
CSV_PATH = Path("B:/Projects/Disertatie/Diabetic-Retinopathy-Classifier/datasets/originals/Diabetic_Balanced_Aug/trainLabels.csv")
INPUT_DIR = Path("B:/Projects/Disertatie/Diabetic-Retinopathy-Classifier/datasets/originals/Diabetic_Balanced_Aug/resized_train/resized_train")
OUTPUT_DIR = Path("B:/Projects/Disertatie/Diabetic-Retinopathy-Classifier/datasets/processed_by_me/balanced_aug/balanced_aug_cleaned")

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif"}
CLASSES = ["0", "1", "2", "3", "4"]
WORKERS = max((os.cpu_count() or 2) - 1, 1)

RATIO_TRAIN, RATIO_VAL, RATIO_TEST = 0.70, 0.20, 0.10

# ==========================================
# 2. PRAGURI DINAMICE (Extrem de Agresiv pt 0)
# ==========================================
THRESHOLDS = {
    "0": {
        "blur_min": 45.0,        # Acceptă imagini o idee mai puțin clare
        "bright_min": 15.0,      # Acceptă imagini puțin mai întunecate
        "bright_max": 180.0,     # Acceptă imagini global mai luminoase
        "area_min": 0.25,        # Acceptă ochi fotografiați de un pic mai departe
        "area_max": 0.95,        # Acceptă ochi tăiați puțin mai mult de marginile pozei
        "circularity_min": 0.80, # Permite ochi ovali sau tăiați pe o latură
        "glare_max_ratio": 0.005 # Permite ca blițul camerei să ocupe până la 0.5% din ochi
    },
    "minoritare": {
        "blur_min": 15.0,       
        "bright_min": 8.0,      
        "bright_max": 220.0,    
        "area_min": 0.15,       
        "area_max": 0.98,       
        "circularity_min": 0.60,  # Acceptam ochi taiati masiv sau deformati
        "glare_max_ratio": 0.05   # Acceptam reflexii mari, vrem sa pastram tesutul bolnav
    }
}

# ==========================================
# 3. FUNCTIA DE EVALUARE SI CURATARE
# ==========================================
def evaluate_image(image_path, cls):
    img = cv2.imread(str(image_path))
    if img is None: return False, "Eroare_Citire"

    rules = THRESHOLDS["0"] if str(cls) == "0" else THRESHOLDS["minoritare"]

    img_resized = cv2.resize(img, (512, 512))
    gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
    total_pixels = 512 * 512

    non_black_pixels = gray[gray > 10]
    if len(non_black_pixels) == 0: return False, "Imagine_Neagra"
        
    mean_brightness = np.mean(non_black_pixels)
    if mean_brightness < rules["bright_min"]: return False, "Prea_Intunecata"
    if mean_brightness > rules["bright_max"]: return False, "Supraexpusa_Total"

    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    if laplacian_var < rules["blur_min"]: return False, "Blurata"

    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    kernel_clean = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    thresh_clean = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_clean)
    contours, _ = cv2.findContours(thresh_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours: return False, "Fara_Contur"

    contur_ochi = max(contours, key=cv2.contourArea)
    ochi_area = cv2.contourArea(contur_ochi)
    
    area_ratio = ochi_area / total_pixels
    if area_ratio < rules["area_min"]: return False, "Zoom_Prea_Mic"
    if area_ratio > rules["area_max"]: return False, "Zoom_Exagerat_Taiat"

    perimeter = cv2.arcLength(contur_ochi, True)
    if perimeter == 0: return False, "Eroare_Geometrie"
        
    circularity = (4 * np.pi * ochi_area) / (perimeter * perimeter)
    if circularity < rules["circularity_min"]: return False, "Forma_Taiata_Neregulata"

    glare_mask = gray > 245
    glare_ratio = np.sum(glare_mask) / (ochi_area + 1e-6) 
    
    if glare_ratio > rules["glare_max_ratio"]: return False, "Reflexie_Blit_Lentila"

    return True, "OK"

def process_evaluation_task(task):
    img_path, label = task
    is_valid, reason = evaluate_image(img_path, label)
    return img_path, label, is_valid, reason

def copy_file(task):
    src, dst = task
    try:
        shutil.copy2(src, dst)
        return True
    except: return False

# ==========================================
# 4. EXECUTIA PRINCIPALA
# ==========================================
def main():
    if not CSV_PATH.exists() or not INPUT_DIR.exists():
        print("Eroare: Calea catre CSV sau INPUT_DIR nu este corecta. Verifica te rog caile.")
        return

    if OUTPUT_DIR.exists():
        print("Curatam folderul de output existent...")
        shutil.rmtree(OUTPUT_DIR)

    for split in ['train', 'val', 'test']:
        for cls in CLASSES:
            (OUTPUT_DIR / split / cls).mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(CSV_PATH)

    print("\n--- 1. Maparea imaginilor de pe disc ---")
    fisiere_existente = {f.stem: f for f in INPUT_DIR.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS}
    
    tasks_eval = []
    # In 'trainLabels.csv' coloanele sunt 'image' (fara extensie) si 'level'
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Asociere fisier"):
        img_id, label = str(row['image']), str(row['level'])
        
        if img_id in fisiere_existente:
            tasks_eval.append((fisiere_existente[img_id], label))

    print(f"\n--- 2. Curatarea imaginilor ({len(tasks_eval)} gasite) ---")
    valid_data = []
    stats_respinse = {}
    
    with ThreadPoolExecutor(max_workers=WORKERS) as executor:
        for result in tqdm(executor.map(process_evaluation_task, tasks_eval), total=len(tasks_eval), desc="Evaluare Filtre"):
            img_path, label, is_valid, reason = result
            if is_valid:
                valid_data.append({'image_path': img_path, 'label': label})
            else:
                stats_respinse[reason] = stats_respinse.get(reason, 0) + 1

    df_valid = pd.DataFrame(valid_data)
    print(f"\n✅ Imagini ramase dupa curatare: {len(df_valid)}")
    print("Motive respingere (Gunoi eliminat):")
    for motiv, count in sorted(stats_respinse.items(), key=lambda x: x[1], reverse=True):
        print(f"  - {motiv}: {count} imagini")

    if len(df_valid) == 0: 
        print("Nu a mai ramas nicio imagine dupa filtrare!")
        return

    print("\n--- 3. Impartirea Stratificata (70/20/10) ---")
    df_train_val, df_test = train_test_split(df_valid, test_size=RATIO_TEST, stratify=df_valid['label'], random_state=42)
    val_fraction = RATIO_VAL / (RATIO_TRAIN + RATIO_VAL)
    df_train, df_val = train_test_split(df_train_val, test_size=val_fraction, stratify=df_train_val['label'], random_state=42)

    print(f"Distributie: TRAIN: {len(df_train)} | VAL: {len(df_val)} | TEST: {len(df_test)}")

    tasks_copy = []
    def add_copy_tasks(dataframe, split_name):
        for _, row in dataframe.iterrows():
            src = row['image_path']
            dst = OUTPUT_DIR / split_name / row['label'] / src.name
            tasks_copy.append((src, dst))

    add_copy_tasks(df_train, 'train')
    add_copy_tasks(df_val, 'val')
    add_copy_tasks(df_test, 'test')

    print("\n--- 4. Copierea imaginilor curate in noua structura ---")
    with ThreadPoolExecutor(max_workers=WORKERS) as executor:
        for _ in tqdm(executor.map(copy_file, tasks_copy), total=len(tasks_copy), desc="Copiere"): pass

    print("\n" + "="*50)
    print("FINALIZAT! Datasetul Balanced_Aug curatat a fost creat cu succes.")
    print(f"Locatia: {OUTPUT_DIR}")
    print("="*50)

if __name__ == '__main__':
    main()

Curatam folderul de output existent...

--- 1. Maparea imaginilor de pe disc ---


Asociere fisier: 100%|██████████| 35126/35126 [00:00<00:00, 40226.82it/s]



--- 2. Curatarea imaginilor (35126 gasite) ---


Evaluare Filtre: 100%|██████████| 35126/35126 [01:13<00:00, 480.57it/s]



✅ Imagini ramase dupa curatare: 20275
Motive respingere (Gunoi eliminat):
  - Blurata: 13653 imagini
  - Reflexie_Blit_Lentila: 610 imagini
  - Supraexpusa_Total: 328 imagini
  - Forma_Taiata_Neregulata: 243 imagini
  - Prea_Intunecata: 10 imagini
  - Imagine_Neagra: 3 imagini
  - Zoom_Exagerat_Taiat: 2 imagini
  - Zoom_Prea_Mic: 2 imagini

--- 3. Impartirea Stratificata (70/20/10) ---
Distributie: TRAIN: 14192 | VAL: 4055 | TEST: 2028

--- 4. Copierea imaginilor curate in noua structura ---


Copiere: 100%|██████████| 20275/20275 [00:06<00:00, 3188.68it/s]


FINALIZAT! Datasetul Balanced_Aug curatat a fost creat cu succes.
Locatia: B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\processed_by_me\balanced_aug\balanced_aug_cleaned
